In [ ]:

import torch
import torch.nn as nn

# Simple demo model (Layered MLP)
class DemoTransformerLayer(nn.Module):
    def __init__(self, d_model=1024):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_model * 4, bias=False)
        self.fc2 = nn.Linear(d_model * 4, d_model, bias=False)

    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))

def simulate_zero_memory(world_size=32, d_model=1024):
    model = DemoTransformerLayer(d_model=d_model)
    total_params = sum(p.numel() for p in model.parameters())

    # Memory sizes in Bytes (FP32)
    param_size = total_params * 4
    grad_size = total_params * 4
    adam_state_size = total_params * 4 * 2  # Momentum + Variance

    print(f"--- Model Parameters: {total_params:,} | Full State Size: {(param_size + grad_size + adam_state_size)/1e6:.2f} MB ---\n")

    # ZeRO-1: Optimizer States Partitioned
    z1_total = (param_size + grad_size + (adam_state_size / world_size)) / 1e6

    # ZeRO-2: Optimizer States + Gradients Partitioned
    z2_total = (param_size + (grad_size / world_size) + (adam_state_size / world_size)) / 1e6

    # ZeRO-3: Parameters + Gradients + Optimizer States Partitioned
    z3_at_rest = ((param_size / world_size) + (grad_size / world_size) + (adam_state_size / world_size)) / 1e6
    z3_peak = (param_size + (grad_size / world_size) + (adam_state_size / world_size)) / 1e6

    print(f"[ZeRO-1 Static Memory per Rank]: {z1_total:.2f} MB")
    print(f"[ZeRO-2 Static Memory per Rank]: {z2_total:.2f} MB")
    print(f"[ZeRO-3 Static Memory per Rank (at rest)]: {z3_at_rest:.2f} MB")
    print(f"[ZeRO-3 Peak Memory during layer forward/backward]: {z3_peak:.2f} MB")

# Run directly without spawning processes
simulate_zero_memory(world_size=32)

--- Model Parameters: 8,388,608 | Full State Size: 134.22 MB ---

[ZeRO-1 Static Memory per Rank]: 69.21 MB
[ZeRO-2 Static Memory per Rank]: 36.70 MB
[ZeRO-3 Static Memory per Rank (at rest)]: 4.19 MB
[ZeRO-3 Peak Memory during layer forward/backward]: 36.70 MB
